<a href="https://colab.research.google.com/github/Mans-doc/MDQ/blob/FOR-MANS/ADILLLET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score

In [21]:

# 1. Загрузка данных
df_business = pd.read_parquet('/content/sample_data/business_cards_MDQ.parquet')
df_consumer = pd.read_parquet('/content/sample_data/consumer_cards_MDQ.parquet')
df_merchants = pd.read_parquet('/content/sample_data/merchants_reference.parquet')

# 2. Создаем таргет (целевую переменную): 1 - бизнес, 0 - потребительская
df_business['is_business'] = 1
df_consumer['is_business'] = 0

# 3. Вертикальное объединение транзакций
df_transactions = pd.concat([df_business, df_consumer], ignore_index=True)

# 4. Горизонтальное присоединение справочника мерчантов
# Предположим, что общий ключ называется 'merchant_id' (проверь имя колонки в своем датасете!)
df_full = pd.merge(df_transactions, df_merchants, on='merchant_id', how='left')

print(f"Общее количество строк после корректного объединения: {len(df_full)}")

Общее количество строк после корректного объединения: 12830080
Общее количество строк после корректного объединения: 12830080


In [ ]:
# Быстрый анализ структуры данных
print(df_full.info(show_counts=True))

# Проверка пропусков в процентах
missing_values = df_full.isnull().mean() * 100
print("\nПроцент пропусков по колонкам:\n", missing_values[missing_values > 0])

# Проверка явных дубликатов строк
duplicates_count = df_full.duplicated().sum()
print(f"\nКоличество полных дубликатов строк: {duplicates_count}")

# ПРИВЕДЕНИЕ ТИПОВ (Пример)
# Замени 'transaction_date' на реальное название твоей колонки с датой
if 'transaction_date' in df_full.columns:
    df_full['transaction_date'] = pd.to_datetime(df_full['transaction_date'])

# Категориальные признаки (например, тип операции, MCC код) переводим в string/category
cat_cols = ['mcc_code', 'merchant_category', 'operation_type'] # подставь свои
for col in cat_cols:
    if col in df_full.columns:
        df_full[col] = df_full[col].astype(str)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12830080 entries, 0 to 12830079
Data columns (total 17 columns):
 #   Column                  Non-Null Count     Dtype         
---  ------                  --------------     -----         
 0   transaction_date        12830080 non-null  object        
 1   transaction_timestamp   12830080 non-null  datetime64[ms]
 2   transaction_amount_kzt  12830080 non-null  int64         
 3   mcc_x                   12830080 non-null  object        
 4   merchant_id             12830080 non-null  object        
 5   channel                 12830080 non-null  object        
 6   bank_name               12830080 non-null  object        
 7   country                 12830080 non-null  object        
 8   card_number             12830080 non-null  object        
 9   card_tier               12830080 non-null  object        
 10  tokenized               12830080 non-null  bool          
 11  is_recurring            12830080 non-null  bool          
 12

In [22]:

if duplicates_count > 0:
    df_full = df_full.drop_duplicates().reset_index(drop=True)

if 'amount' in df_full.columns:

    max_threshold = df_full['amount'].quantile(0.999)
    df_full = df_full[(df_full['amount'] > 0) & (df_full['amount'] <= max_threshold)]
for col in cat_cols:
    if col in df_full.columns:
        df_full[col] = df_full[col].fillna('UNKNOWN')


In [23]:
print(df_full.columns.tolist())

['transaction_date', 'transaction_timestamp', 'transaction_amount_kzt', 'mcc_x', 'merchant_id', 'channel', 'bank_name', 'country', 'card_number', 'card_tier', 'tokenized', 'is_recurring', 'is_business', 'merchant_name', 'mcc_y', 'merchant_country', 'recurring_capable']


In [24]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# =====================================================================
# БЛОК 1: FEATURE ENGINEERING (Агрегации на уровне карт)
# =====================================================================
print("--- Блок 1: Feature Engineering ---")

# 1. Основные агрегаты: суммы, чеки, количество, повторяющиеся платежи и активность по времени
df_features = df_full.groupby('card_number').agg(
    is_business=('is_business', 'first'),                     # Таргет
    total_amount=('transaction_amount_kzt', 'sum'),           # Общая сумма транзакций
    tx_count=('transaction_amount_kzt', 'count'),             # Число транзакций
    avg_check=('transaction_amount_kzt', 'mean'),             # Средний чек
    max_check=('transaction_amount_kzt', 'max'),             # Максимальный чек
    unique_days=('transaction_date', 'nunique'),              # Активность по времени (уникальные дни)
    recurring_count=('is_recurring', 'sum')                   # Сколько было регулярных платежей (доля recurring)
).reset_index()

# Дополнительные расчетные фичи
df_features['amount_per_active_day'] = df_features['total_amount'] / df_features['unique_days']
df_features['recurring_ratio'] = df_features['recurring_count'] / df_features['tx_count'] # Доля recurring

# 2. Доля онлайн/оффлайн через колонку 'channel' (если там значения вроде 'online'/'offline')
# Считаем, сколько транзакций каждого типа совершено по карте
df_channel_pivot = df_full.pivot_table(
    index='card_number',
    columns='channel',
    values='transaction_amount_kzt',
    aggfunc='count',
    fill_value=0
).reset_index()
# Переводим в долю (ratio) от общего числа транзакций
for col in df_channel_pivot.columns:
    if col != 'card_number':
        df_channel_pivot[f'channel_ratio_{col}'] = df_channel_pivot[col] / df_full.groupby('card_number')['transaction_amount_kzt'].count().values
        df_channel_pivot.drop(columns=[col], inplace=True)

# 3. Распределение по MCC (Сводная таблица объемов трат по кодам)
df_mcc_pivot = df_full.pivot_table(
    index='card_number',
    columns='mcc_x',
    values='transaction_amount_kzt',
    aggfunc='sum',
    fill_value=0
).reset_index()

# Переименовываем MCC-колонки, чтобы модель понимала их как признаки
df_mcc_pivot.columns = [f"mcc_vol_{col}" if col != 'card_number' else col for col in df_mcc_pivot.columns]

# 4. Объделим все фичи вместе
df_final_dataset = pd.merge(df_features, df_channel_pivot, on='card_number', how='left')
df_final_dataset = pd.merge(df_final_dataset, df_mcc_pivot, on='card_number', how='left')

print(f"Размерность датасета после Feature Engineering: {df_final_dataset.shape}")


# =====================================================================
# БЛОК 2: РАЗДЕЛЕНИЕ НА TRAIN / TEST (Строго по картам)
# =====================================================================
print("\n--- Блок 2: Разделение на Train/Test ---")

# Отделяем матрицу признаков (X) от таргета (y). Номер карты убираем из обучения.
X = df_final_dataset.drop(columns=['card_number', 'is_business'])
y = df_final_dataset['is_business']

# Делаем split. stratify=y гарантирует одинаковый процент бизнеса в обеих выборках
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")


# =====================================================================
# БЛОК 3: КОДИРОВАНИЕ И НОРМАЛИЗАЦИЯ (Защита от Data Leakage)
# =====================================================================
print("\n--- Блок 3: Масштабирование признаков ---")

scaler = StandardScaler()

# Обучаем scaler ТОЛЬКО на Train выборке и трансформируем её
X_train_scaled = scaler.fit_transform(X_train)

# К Test выборке применяем ТУ ЖЕ шкалу (только transform)
X_test_scaled = scaler.transform(X_test)

# Возвращаем данные в формат Pandas DataFrame
X_train_ready = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_ready = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print("🟢 Все три блока успешно выполнены без ошибок!")

--- Блок 1: Feature Engineering ---
Размерность датасета после Feature Engineering: (105000, 544)

--- Блок 2: Разделение на Train/Test ---
Train: (84000, 542), Test: (21000, 542)

--- Блок 3: Масштабирование признаков ---
🟢 Все три блока успешно выполнены без ошибок!
